**P3**

I already scraped my first dataset in P2, P3 wil be about obtaining my second dataset from IMDB. It is a downloadable dataset and then merging it with the dataset I already have. It is downloadable using https://datasets.imdbws.com/title.ratings.tsv.gz

**Deliverable 1**

In [1]:
#Setup
import os
import time
import requests
import pandas as pd

DATA_DIR = "data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

IMDB_URL = "https://datasets.imdbws.com/title.ratings.tsv.gz"
IMDB_FILE = os.path.join(DATA_DIR, "title.ratings.tsv.gz")

headers = {
    "User-Agent": "NCSU DSA 405 coursework data acquisition"
}


def download_file(url, output_path):

    #Use the saved raw file if it already exists
    if os.path.exists(output_path):
        print("Using cached file:", output_path)
        return

    #Wait one second before making the request
    time.sleep(1)

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=30
        )

    except requests.RequestException as error:
        raise RuntimeError(
            f"Request failed: {error}"
        )

    #Stop if the website returns an unsuccessful status code
    if response.status_code != 200:
        raise RuntimeError(
            f"Request returned status code {response.status_code}"
        )

    #Stop if the response contains no data
    if len(response.content) == 0:
        raise RuntimeError(
            "The response contained no data."
        )

    #Save the raw response exactly as it was downloaded
    with open(output_path, "wb") as file:
        file.write(response.content)

    print("Downloaded and saved:", output_path)

In [2]:
#Demonstrate that the code fails loudly on bad requests:
try:
    download_file(
        "https://datasets.imdbws.com/this_file_does_not_exist.tsv.gz",
        "data/raw/test_failure.tsv.gz"
    )

except RuntimeError as error:
    print("Expected failure caught:")
    print(error)

Expected failure caught:
Request returned status code 404


In [3]:
#Retrieve the data
download_file(
    IMDB_URL,
    IMDB_FILE
)

imdb_ratings = pd.read_csv(
    IMDB_FILE,
    sep="\t",
    compression="gzip"
)

Using cached file: data/raw/title.ratings.tsv.gz


**Deliverable 2**

| A8 plan item | Final P3 approach | Match / Drift |
|---|---|---|
| Use IMDb's official `title.ratings.tsv.gz` dataset | Used the same official IMDb ratings dataset | Match |
| Direct download rather than website scraping | Downloaded the compressed TSV file directly from `datasets.imdbws.com` | Match |
| No login or paywall | Dataset is publicly downloadable without an account or payment | Match |
| No personal data | Ratings file contains title IDs, aggregated ratings, and vote counts rather than individual user information | Match |
| No circumvention | No login, paywall, bot protection, or other access restriction was bypassed | Match |
| Honest User-Agent | Request identifies itself as NCSU DSA 405 coursework | Match |
| Rate-limited requests | Requests are separated by at least one second | Match |
| Cache responses | The saved raw file is reused instead of downloading the same resource again | Match |
| Reliable failure handling | Added timeout, non-200 response, empty-response, and fail-loud checks to satisfy the P3 acquisition requirements | Drift |


The last P3 acquisition process is the same source and approach that has been stated in A8. The only difference is the drift in failure handling, where there was a need to check for timeout, non-200 status codes, and no-response situations. This was done to satisfy the requirements of P3 acquisition and not alter the source and access process in any way.



**Deliverable 3**

In [4]:
#Check that there is a unique number of imdb ids

print(imdb_ratings["tconst"].duplicated().sum())

print(imdb_ratings["tconst"].count())

print(imdb_ratings.columns)

0
1727597
Index(['tconst', 'averageRating', 'numVotes'], dtype='object')


|                                   |                                                       |
| --------------------------------- | ----------------------------------------------------- |
| **Join key(s)**                   | TMDB `imdb_id` = IMDb `tconst`                        |
| **Grain of the left table**       | "one row per movie"                                     |
| **Grain of the right table**      | "one row per IMDB title ('tconst')"                                     |
| **Relationship**                  | one-to-one |
| **Expected row count after the join** | 5810                                         |
| **Join type and why**             | Left join, because all cleaned TMDB movies should remain in the dataset even if no matching IMDb rating is found                         |
| **What happens to unmatched rows**| Unmatched TMDB rows are kept, and the IMDb fields are filled with NA                       |

In [5]:
#Retrieve the left dataset

CLEAN_DIR = "data/clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

TMDB_CLEAN_FILE = os.path.join(CLEAN_DIR, "tmdb_clean.json")


tmdb_clean = pd.read_json(TMDB_CLEAN_FILE)

print(tmdb_clean.shape)
display(tmdb_clean.head())

(5810, 11)


,id,imdb_id,title,release_date,genres,budget,revenue,roi,top_actor_name,max_actor_popularity,top_3_actor_popularity
0,98,tt0172495,Gladiator,2000-05-04T00:00:00.000,"[Action, Drama, Adventure]",103000000,465516248,3.519575,Russell Crowe,4.9597,4.402733
1,4247,tt0175142,Scary Movie,2000-07-07T00:00:00.000,[Comedy],19000000,278023280,13.632804,Lochlyn Munro,4.4736,3.770633
2,107,tt0208092,Snatch,2000-09-01T00:00:00.000,"[Crime, Comedy]",100000000,83557872,-0.164421,Jason Statham,16.7122,10.520733
3,1359,tt0144084,American Psycho,2000-04-13T00:00:00.000,"[Thriller, Drama, Crime]",7000000,34269748,3.895678,Willem Dafoe,5.8196,4.797167
4,11688,tt0120917,The Emperor's New Groove,2000-12-15T00:00:00.000,"[Adventure, Animation, Comedy, Family, Fantasy]",100000000,169327687,0.693277,John Goodman,3.6030,3.038533


In [6]:
#Check shape

print(tmdb_clean.shape)
print(imdb_ratings.shape)

#Check for missing values
print("Missing IMDb IDs:", tmdb_clean["imdb_id"].isna().sum())

print(
    "Duplicate non-missing IMDb IDs:",
    tmdb_clean["imdb_id"].dropna().duplicated().sum()
)


(5810, 11)
(1727597, 3)
Missing IMDb IDs: 26
Duplicate non-missing IMDb IDs: 0


In [7]:
#Separate movies with and without IMDb IDs
tmdb_with_id = tmdb_clean[tmdb_clean["imdb_id"].notna()]
tmdb_without_id = tmdb_clean[tmdb_clean["imdb_id"].isna()].copy()

# Merge movies that have IMDb IDs
movie_data_with_id = tmdb_with_id.merge(
    imdb_ratings,
    left_on="imdb_id",
    right_on="tconst",
    how="left",
    validate="one_to_one"
)

#Add empty IMDb columns to movies that do not have IMDb IDs
tmdb_without_id["tconst"] = pd.NA
tmdb_without_id["averageRating"] = pd.NA
tmdb_without_id["numVotes"] = pd.NA

#Put all movies back together
movie_data = pd.concat(
    [movie_data_with_id, tmdb_without_id],
    ignore_index=True
)

print(movie_data.shape)
print(movie_data.columns)

movie_data["release_date"] = pd.to_datetime(
    movie_data["release_date"],
    errors="coerce"
)

movie_data.to_csv(
    "data/clean/movie_data.csv",
    index=False
)

(5810, 14)
Index(['id', 'imdb_id', 'title', 'release_date', 'genres', 'budget', 'revenue',
       'roi', 'top_actor_name', 'max_actor_popularity',
       'top_3_actor_popularity', 'tconst', 'averageRating', 'numVotes'],
      dtype='object')


/tmp/ipykernel_45947/3484547279.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  movie_data = pd.concat(


The validate="one_to_one" argument guards against fan-out by stopping the merge if duplicate join keys would cause rows to be added unexpectedly. Using a left join guards against the inner-join “wreck” because all 5,810 rows from the cleaned TMDB dataset are preserved even when no matching IMDb rating is found. The 26 TMDB rows with missing IMDb IDs were handled separately by creating one dataset with the missing rows and one without. The TMDMB dataset and the dataset without the missing imdb_id rows were merged, and then the dataset with the missing imdb_id values was concatted back to the merged dataset to retain the valuable information.

**Deliverable 4**



In [8]:
#protects against: impossible negative IMDb vote counts
assert (movie_data["numVotes"].dropna() >= 0).all()

#protects against: IMDb rating data being only partially attached to a movie
assert (
    movie_data["averageRating"].notna()
    == movie_data["numVotes"].notna()
).all()

#protects against: movie release dates falling outside the 2000-2025 collection period
assert movie_data["release_date"].dropna().dt.year.between(2000, 2025).all()

All the assertions passed. One thing that surprised me was that the dataset couldn't merge if the any of the grains had missing values. At first when I merged, the left grain had 26 missing values. At first, I thought they would just automatically merge and be added as NaN, but then I learned that I had to create a new dataset without those missing values and then add the rows that hadd the missing left grain values after I finished the completed merge.